# ✦ VLM UI Generator
> **Sketch → Polished Mobile UI** · CLIP + Stable Diffusion + ControlNet

### ⚡ Quick Start
1. `Runtime → Change runtime type → GPU (T4)`
2. Run **Setup** cells (once per session)
3. Run **Train** OR jump straight to **Launch Web UI** to try demo mode

---

## 🔧 Step 1 — Environment Setup

In [ ]:
# Check GPU
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')

In [ ]:
# Mount Google Drive to persist checkpoints across sessions
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/vlm_ui_generator'
import os
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Drive mounted. Project dir:', DRIVE_DIR)

In [ ]:
%%capture
# Install all dependencies (~3 mins on first run)
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install transformers diffusers accelerate peft datasets \
             gradio omegaconf einops kagglehub safetensors xformers \
             opencv-python-headless scikit-learn matplotlib seaborn \
             sentencepiece huggingface-hub
print('✅ Dependencies installed')

In [ ]:
import os, shutil

PROJECT_DIR = '/content/vlm_ui_generator'

# Option A: Upload zip (if you downloaded from Claude)
# from google.colab import files
# uploaded = files.upload()  # upload vlm_ui_generator.zip
# !unzip -q vlm_ui_generator.zip -d /content/

# Option B: Clone from your GitHub (recommended)
# !git clone https://github.com/YOUR_USERNAME/vlm_ui_generator /content/vlm_ui_generator

# Option C: Use this cell to create the project structure programmatically
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
print('Working directory:', os.getcwd())

# Symlink checkpoints to Drive so they persist
ckpt_drive = f'{DRIVE_DIR}/checkpoints'
os.makedirs(ckpt_drive, exist_ok=True)
ckpt_local = f'{PROJECT_DIR}/checkpoints'
if not os.path.exists(ckpt_local):
    os.symlink(ckpt_drive, ckpt_local)
print('Checkpoints linked to Drive:', ckpt_drive)

## 🔑 Step 2 — Kaggle API Credentials

In [ ]:
# Set your Kaggle credentials to download datasets
import os, json

# Paste your credentials from kaggle.com → Account → Create API Token
KAGGLE_USERNAME = 'YOUR_KAGGLE_USERNAME'  # ← change this
KAGGLE_KEY      = 'YOUR_KAGGLE_API_KEY'   # ← change this

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
creds = {'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    json.dump(creds, f)
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
os.environ['KAGGLE_KEY']      = KAGGLE_KEY
print('✅ Kaggle credentials configured')

## 🏋️ Step 3 — Train the Model

In [ ]:
# ── Training parameters (edit freely) ────────────────────────
BATCH_SIZE    = 4      # Lower if you get OOM errors
NUM_EPOCHS    = 30
PHASE1_EPOCHS = 10     # Epochs with encoders frozen
LR            = 1e-4
USE_FP16      = True   # Strongly recommended on T4

import subprocess, sys

cmd = [
    sys.executable, 'train.py',
    f'training.batch_size={BATCH_SIZE}',
    f'training.num_epochs={NUM_EPOCHS}',
    f'training.phase1_epochs={PHASE1_EPOCHS}',
    f'optimizer.lr={LR}',
    f'training.mixed_precision={"fp16" if USE_FP16 else "no"}',
    'training.gradient_checkpointing=true',
]

print('Starting training with command:')
print(' '.join(cmd))
print('─' * 60)

# Runs synchronously — output streams to this cell
result = subprocess.run(cmd)
print('Training exit code:', result.returncode)

In [ ]:
# Plot loss curves after training
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

log_path = Path('outputs/logs/train_log.csv')
if log_path.exists():
    df = pd.read_csv(log_path)
    fig, axes = plt.subplots(2, 2, figsize=(12, 7), facecolor='#0f0f19')
    for ax, col, title, c in zip(
        axes.flatten(),
        ['loss_total','loss_gen','loss_rating','loss_align'],
        ['Total','Generation','Rating','Alignment'],
        ['#6366f1','#10b981','#f59e0b','#ec4899'],
    ):
        ax.set_facecolor('#1a1a2e')
        if col in df.columns:
            ax.plot(df['step'], df[col], color=c, lw=1.8)
            ax.fill_between(df['step'], df[col], alpha=0.15, color=c)
        ax.set_title(f'{title} Loss', color='white')
        ax.tick_params(colors='gray')
    plt.suptitle('Training Loss Curves', color='white', fontsize=13)
    plt.tight_layout()
    plt.savefig('outputs/logs/loss_curves.png', dpi=130, bbox_inches='tight', facecolor='#0f0f19')
    plt.show()
else:
    print('No training log found yet. Run training first.')

## 🌐 Step 4 — Launch Web Interface

In [ ]:
# ── Launch the Gradio web app ─────────────────────────────────
# share=True creates a public URL you can open from any browser,
# including your phone! The link is valid for 72 hours.

import subprocess, sys, os

os.chdir('/content/vlm_ui_generator')

proc = subprocess.Popen(
    [sys.executable, 'app.py', '--share'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

# Stream output until we find the public URL
for line in proc.stdout:
    print(line, end='')
    if 'gradio.live' in line or 'Running on' in line:
        break  # URL found — keep app running in background

print('\n✅ Web app is running! Open the URL above in any browser.')
print('The Colab session keeps it alive — do not close this tab.')

## 🎨 Step 5 — Quick Generate (without web UI)
Use this cell to generate a single UI image directly in the notebook.

In [ ]:
from PIL import Image
import numpy as np

# ── Option A: Load a real sketch ───────────────────────────────
# from google.colab import files
# uploaded = files.upload()
# sketch = Image.open(list(uploaded.keys())[0]).convert('RGB')

# ── Option B: Generate a synthetic test sketch ─────────────────
import cv2
sketch_arr = np.ones((512, 512, 3), dtype=np.uint8) * 255
# Draw simple wireframe boxes (simulate a sketch)
cv2.rectangle(sketch_arr, (40, 40), (472, 80), (0,0,0), 2)    # header
cv2.rectangle(sketch_arr, (40, 100), (472, 160), (0,0,0), 2)  # search bar
for i in range(3):
    y = 180 + i * 100
    cv2.rectangle(sketch_arr, (40, y), (472, y+80), (0,0,0), 2)
cv2.rectangle(sketch_arr, (0, 450), (512, 512), (0,0,0), 2)   # bottom nav
sketch = Image.fromarray(sketch_arr)

# ── Your prompt ────────────────────────────────────────────────
PROMPT = "Design a clean mobile home screen with a search bar and content cards"

# ── Generate ───────────────────────────────────────────────────
try:
    from inference.inference_engine import InferenceEngine
    engine = InferenceEngine.from_checkpoint(
        checkpoint_path   = 'checkpoints/best_model.pth',
        model_config_path = 'config/model_config.yaml',
        device            = 'cuda',
    )
    result = engine.generate(sketch_image=sketch, prompt=PROMPT, seed=42)
    print('✅ Generated from trained model')
except Exception as e:
    # Demo mode: generates a styled placeholder
    print(f'Demo mode (no checkpoint): {e}')
    from app import _demo_generate
    result = _demo_generate(sketch, PROMPT, seed=42)

# Show side by side
import matplotlib.pyplot as plt
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5), facecolor='#0f0f19')
for ax, img, title in [(ax1, sketch, 'Input Sketch'), (ax2, result, 'Generated UI')]:
    ax.imshow(img)
    ax.set_title(title, color='white', pad=8)
    ax.axis('off')
    ax.set_facecolor('#0f0f19')
plt.suptitle(PROMPT[:70], color='#a5b4fc', fontsize=9)
plt.tight_layout()
plt.savefig('outputs/generated/colab_result.png', dpi=130,
            bbox_inches='tight', facecolor='#0f0f19')
plt.show()
print('Saved to outputs/generated/colab_result.png')

## 💾 Save Checkpoint to Drive

In [ ]:
import shutil, glob
ckpts = sorted(glob.glob('checkpoints/*.pth'))
if ckpts:
    best = 'checkpoints/best_model.pth'
    if os.path.exists(best):
        shutil.copy(best, f'{DRIVE_DIR}/best_model.pth')
        print('✅ Best model saved to Drive:', f'{DRIVE_DIR}/best_model.pth')
    for c in ckpts:
        shutil.copy(c, f'{DRIVE_DIR}/{os.path.basename(c)}')
    print(f'✅ All {len(ckpts)} checkpoints backed up to Drive')
else:
    print('No checkpoints found yet.')